# AbLang -- Antibody Language Model

This notebook demonstrates how to use **AbLang** for antibody-specific embeddings, sequence scoring, and masked position restoration.

## What is AbLang?

AbLang is a family of antibody-specific language models trained on antibody sequence data from the Observed Antibody Space (OAS). It provides:

- **Embeddings** -- Antibody representations for similarity search, clustering, and downstream ML
- **Scoring** -- Pseudo-log-likelihood scoring to assess antibody sequence naturalness
- **Restore** -- Predict masked positions in antibody sequences (e.g., CDR loop reconstruction)

**Links:** [Paper (DOI: 10.1093/bioadv/vbae040)](https://doi.org/10.1093/bioadv/vbae040) | [GitHub](https://github.com/oxpig/AbLang2)

## Model Variants

AbLang wraps two generations of models under a single tool. The `model_choice` config field controls which variant is used:

| `model_choice` | Model | Input Format | Embedding Dim | When to Use |
|---|---|---|---|---|
| `"ablang1-heavy"` | AbLang1 heavy | Single heavy chain | 768 | VH-only analysis |
| `"ablang1-light"` | AbLang1 light | Single light chain | 768 | VL-only analysis |
| `"ablang2-paired"` | AbLang2 | Paired `"heavy\|light"` | 480 | Paired VH+VL analysis |

## Automatic Model Routing

By default, `model_choice="auto"`. The tool inspects the input sequences and picks the right model:

- If **any** sequence contains a `|` separator → routes to **`ablang2-paired`**
- Otherwise → routes to **`ablang1-heavy`** (the most common single-chain use case)

To use `ablang1-light`, you must set `model_choice="ablang1-light"` explicitly.

## Imports

In [3]:
from proto_tools.tools.masked_models.ablang import (
    run_ablang_embeddings,
    AbLangEmbeddingsInput,
    AbLangEmbeddingsConfig,
    run_ablang_score,
    AbLangScoringInput,
    AbLangScoringConfig,
    run_ablang_sample,
    AbLangSampleInput,
    AbLangSampleConfig,
)

## Input API Reference

### `AbLangEmbeddingsInput`

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Antibody sequence(s) to embed. For paired models, use `"heavy\|light"` format. |

### `AbLangScoringInput`

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Antibody sequence(s) to score. For paired models, use `"heavy\|light"` format. |

### `AbLangSampleInput`

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Antibody sequence(s) with `_` at positions to restore. For paired models, use `"heavy\|light"` format. |

## Config API Reference

### `AbLangEmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_choice` | `str` | `"auto"` | Model variant: `auto`, `ablang1-heavy`, `ablang1-light`, or `ablang2-paired` |
| `batch_size` | `int` | `1` | Number of sequences per forward pass |
| `device` | `str` | `"cuda"` | Device to run the model on |

### `AbLangScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_choice` | `str` | `"auto"` | Model variant: `auto`, `ablang1-heavy`, `ablang1-light`, or `ablang2-paired` |
| `batch_size` | `int` | `1` | Number of sequences per forward pass |
| `device` | `str` | `"cuda"` | Device to run the model on |
| `scoring_mode` | `str` | `"pseudo_log_likelihood"` | `"pseudo_log_likelihood"` (accurate, slower) or `"confidence"` (fast) |

### `AbLangSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_choice` | `str` | `"auto"` | Model variant: `auto`, `ablang1-heavy`, `ablang1-light`, or `ablang2-paired` |
| `batch_size` | `int` | `1` | Number of sequences per forward pass |
| `device` | `str` | `"cuda"` | Device to run the model on |

## Output API Reference

### `AbLangEmbeddingsOutput`

| Field | Type | Description |
|-------|------|-------------|
| `results` | `List[SequenceEmbedding]` | Per-sequence embedding results |

Each `SequenceEmbedding` contains:

| Field | Type | Description |
|-------|------|-------------|
| `mean_embedding` | `List[float]` | Mean-pooled embedding vector (768-dim for ablang1, 480-dim for ablang2-paired) |
| `attention_mask` | `List[int]` | Binary mask: 1 = valid position, 0 = padding |
| `logits` | `Optional[List[List[float]]]` | Per-position amino acid logits (always `None` for AbLang) |

### `AbLangScoringOutput` (`MaskedModelScoringOutput`)

| Field | Type | Description |
|-------|------|-------------|
| `scores` | `List[SequenceScores]` | Per-sequence scoring results |

Each `SequenceScores` contains:

| Field | Type | Description |
|-------|------|-------------|
| `metrics` | `Dict[str, float]` | Scoring metrics (e.g., `pseudo_log_likelihood`) |
| `logits` | `Optional[List[List[float]]]` | Per-position logits (`None` for scoring) |
| `vocab` | `Optional[List[str]]` | Token ordering for logits (`None` for scoring) |

### `AbLangSampleOutput`

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Restored antibody sequences with masked positions filled |

## 1. Extract Embeddings (auto-routing)

With `model_choice="auto"` (the default), the tool detects single-chain vs paired sequences automatically. Here we pass single heavy chains, so the tool routes to `ablang1-heavy` and returns 768-dim embeddings.

In [8]:
# Trastuzumab (Herceptin) VH sequence
trastuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"

# Pertuzumab VH sequence (another anti-HER2 antibody)
pertuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFTFTDYTMDWVRQAPGKGLEWVADVNPNSGGSIYNQRFKGRFTLSVDRSKNTLYLQMNSLRAEDTAVYYCARNLGPSFYFDYWGQGTLVTVSS"

# No model_choice specified — defaults to "auto", which routes single chains to ablang1-heavy
inputs = AbLangEmbeddingsInput(sequences=[trastuzumab_vh, pertuzumab_vh])
config = AbLangEmbeddingsConfig(batch_size=2)

embeddings_result = run_ablang_embeddings(inputs, config)

print(f"Model used: {embeddings_result.metadata['model_choice']}")
print(f"Processed {len(embeddings_result.results)} sequences")
print(f"Embedding dimension: {len(embeddings_result.results[0].mean_embedding)}")
print(f"Attention mask length (trastuzumab): {sum(embeddings_result.results[0].attention_mask)}")
print(f"Attention mask length (pertuzumab): {sum(embeddings_result.results[1].attention_mask)}")

Model used: ablang1-heavy
Processed 2 sequences
Embedding dimension: 768
Attention mask length (trastuzumab): 120
Attention mask length (pertuzumab): 119


## 2. Score Sequences

Compute pseudo-log-likelihood scores to assess how natural an antibody sequence looks to the model. Lower (more negative) scores indicate lower model confidence. This is useful for comparing natural antibodies against engineered or scrambled variants.

Again using auto-routing here — single chains route to `ablang1-heavy`.

In [10]:
# Compare a natural antibody heavy chain vs a scrambled version
natural_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
# Scrambled version (same amino acids, random order — no non-standard residues)
scrambled_vh = "GVSTYGDRGYPQFSVWVDTAQYNRGISGLKRAAKEGSTYTIRRFLPVDSANDWTYGLPHVSGMTEQNLVCSAYGIKARLWPGNVTSELDFAGTKYREPCGQIHALNMSSWGTDAVYAEV"

# Auto-routes to ablang1-heavy (single chains, no "|")
inputs = AbLangScoringInput(sequences=[natural_vh, scrambled_vh])
config = AbLangScoringConfig(
    scoring_mode="pseudo_log_likelihood",
    batch_size=2,
)

score_result = run_ablang_score(inputs, config)

print("Pseudo-log-likelihood scores (higher = more natural):")
print(f"  Natural VH:   {score_result.scores[0].metrics['pseudo_log_likelihood']:.3f}")
print(f"  Scrambled VH:  {score_result.scores[1].metrics['pseudo_log_likelihood']:.3f}")

Pseudo-log-likelihood scores (higher = more natural):
  Natural VH:   -0.823
  Scrambled VH:  -3.206


## 3. Restore Masked Positions

Use AbLang to predict amino acids at masked positions (marked with `_`). This is especially useful for restoring CDR3 regions or filling in missing residues in antibody sequences.

In [12]:
# Trastuzumab VH with CDR3 region masked (positions 99-110: SRWGGDGFYAMDY)
masked_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYC_____________WGQGTLVTVSS"

print(f"Original CDR3: SRWGGDGFYAMDY")
print(f"Masked input:  ...TAVYYC_____________WGQGTL...")

# Auto-routes to ablang1-heavy (single chain)
inputs = AbLangSampleInput(sequences=[masked_vh])
sample_result = run_ablang_sample(inputs)

restored_seq = sample_result.sequences[0]
# Extract the restored CDR3 region
restored_cdr3 = restored_seq[97:110]
print(f"Restored CDR3: {restored_cdr3}")
print(f"Full restored: {restored_seq}")

Original CDR3: SRWGGDGFYAMDY
Masked input:  ...TAVYYC_____________WGQGTL...
Restored CDR3: RDGGGGGYYFDYW
Full restored: EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARDGGGGGYYFDYWGQGTLVTVSS


## 4. Paired Heavy|Light Mode (auto-routing to ablang2-paired)

When sequences contain a `|` separator, auto-routing selects `ablang2-paired`, which processes heavy and light chains together to capture inter-chain dependencies. The paired model produces 480-dim embeddings instead of 768-dim.

In [14]:
# Trastuzumab paired heavy|light chains
trastuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
trastuzumab_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"

paired_sequence = f"{trastuzumab_vh}|{trastuzumab_vl}"

# The "|" in the sequence triggers auto-routing to ablang2-paired
inputs = AbLangEmbeddingsInput(sequences=[paired_sequence])
paired_result = run_ablang_embeddings(inputs)
print(f"Model used: {paired_result.metadata['model_choice']}")
print(f"Paired embedding dimension: {len(paired_result.results[0].mean_embedding)}")

# Scoring — also auto-routes to ablang2-paired
inputs = AbLangScoringInput(sequences=[paired_sequence])
config = AbLangScoringConfig(scoring_mode="pseudo_log_likelihood")
paired_score = run_ablang_score(inputs, config)
print(f"Paired PLL score: {paired_score.scores[0].metrics['pseudo_log_likelihood']:.3f}")

# Restore masked positions in paired mode (mask CDR3 of light chain)
masked_paired = f"{trastuzumab_vh}|DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYC_________FGQGTKVEIK"
inputs = AbLangSampleInput(sequences=[masked_paired])
paired_restore = run_ablang_sample(inputs)
restored_heavy, restored_light = paired_restore.sequences[0].split("|")
print(f"Original VL CDR3: QQHYTTPPT")
print(f"Restored VL:      {restored_light}")

Model used: ablang2-paired
Paired embedding dimension: 480
Paired PLL score: -0.934
Original VL CDR3: QQHYTTPPT
Restored VL:      DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQANSYPWTFGQGTKVEIK


## 5. Explicit Model Selection (light chain)

Auto-routing defaults single chains to `ablang1-heavy`. For light chains, set `model_choice="ablang1-light"` explicitly. This is the only case where you need to override the default.

In [16]:
# Light chain sequence — must explicitly set model_choice="ablang1-light"
trastuzumab_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"

inputs = AbLangEmbeddingsInput(sequences=[trastuzumab_vl])
config = AbLangEmbeddingsConfig(model_choice="ablang1-light")

light_result = run_ablang_embeddings(inputs, config)

print(f"Model used: {light_result.metadata['model_choice']}")
print(f"Embedding dimension: {len(light_result.results[0].mean_embedding)}")

Model used: ablang1-light
Embedding dimension: 768


## 6. Export Results

Save results to files for downstream analysis.

### Supported export formats

| Output type | Formats |
|---|---|
| Embeddings | `csv`, `json`, `npy`, `pt` |
| Scoring | `csv`, `json` |
| Sampling | `fasta`, `txt`, `json` |

In [18]:
# Export embeddings to CSV
embeddings_result.export("ablang_embeddings", export_path="./example_output", file_format="csv")
print("Embeddings exported to ./example_output/ablang_embeddings.csv")

# Export scores to JSON
score_result.export("ablang_scores", export_path="./example_output", file_format="json")
print("Scores exported to ./example_output/ablang_scores.json")

# Export restored sequences to FASTA
sample_result.export("ablang_restored", export_path="./example_output", file_format="fasta")
print("Restored sequences exported to ./example_output/ablang_restored.fasta")

Embeddings exported to ./example_output/ablang_embeddings.csv
Scores exported to ./example_output/ablang_scores.json
Restored sequences exported to ./example_output/ablang_restored.fasta
